# Classical Models Comparison

Compares generated financial time-series from three models against a common reference:
- **Reference** — ground-truth log-returns from EDM evaluation windows (unnormalized)
- **EDM** — diffusion model samples (unnormalized from z-score)
- **GARCH** — GARCH-X samples (raw log-returns)
- **Merton** — Merton jump-diffusion samples (raw log-returns)

**Part 1** compares aggregate and distributional properties across all four datasets.
**Part 2** compares predictive performance on matched windows (same ticker, same window_start).

In [1]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
dataset = 'train'   # 'train' or 'validation'

# EDM checkpoint sub-directory containing:
#   {dataset}_generated_close.csv  and  {dataset}_gt_ohlc.csv
GEN_DIRECTORY = (
    './data/generated/edm/'
    'EDM_REPL__CLOS_ep-99_step-6534_lr-8e-04_ch-128_layers-6_nheads-4_20260514_201539/'
    'csv500_samples50_seed50_20260515_134544'
)

# Classical model generated CSV paths
GARCH_DIRECTORY  = './data/generated/GARCH/generated_close.csv'
MERTON_DIRECTORY = './data/generated/merton/generated_close.csv'

# Parquet cache paths (used when extract_garch_merton_df = False)
extract_garch_merton_df = False
GARCH_PARQUET  = './data/generated/GARCH/generated_close.parquet'
MERTON_PARQUET = './data/generated/merton/generated_close.parquet'

# Normalization stats used to unnormalize EDM values
# CSV with index = feature name, columns = [mean, std]
UNNORMALIZATION_STATS_PATH = (
    './data/general/normalization_stats_SNP500_super_normalized.csv'
)

# Output directories (created automatically)
OUTPUT_IMAGE_DIR = './images/generated/classical_comparison'
OUTPUT_TABLE_DIR = './tables/generated/classical_comparison'
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve()))
import replication.stylized_facts as sf

warnings.filterwarnings('ignore', category=RuntimeWarning)

Path(OUTPUT_IMAGE_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_TABLE_DIR).mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi'    : 120,
    'font.size'     : 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'lines.linewidth': 1.4,
})

MODEL_COLORS = {
    'Reference': 'tab:blue',
    'EDM'      : 'tab:orange',
    'GARCH'    : 'tab:green',
    'Merton'   : 'tab:red',
}
MODEL_ORDER = ['Reference', 'EDM', 'GARCH', 'Merton']

In [3]:
# ── Utility functions ─────────────────────────────────────────────────────────

def extract_ticker(fname):
    """'AAPL_1980-12-12_2026-05-01.csv' -> 'AAPL'"""
    return str(fname).split('_')[0]


def compute_moments(x, label):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {'model': label}
    return {
        'model'           : label,
        'n'               : len(x),
        'mean'            : np.mean(x),
        'std'             : np.std(x),
        'variance'        : np.var(x),
        'skewness'        : float(scipy_stats.skew(x)),
        'excess_kurtosis' : float(scipy_stats.kurtosis(x, fisher=True)),
        'IQR'             : float(np.quantile(x, 0.75) - np.quantile(x, 0.25)),
        'min'             : np.min(x),
        'q001'            : np.quantile(x, 0.001),
        'q01'             : np.quantile(x, 0.01),
        'q05'             : np.quantile(x, 0.05),
        'q25'             : np.quantile(x, 0.25),
        'q50'             : np.quantile(x, 0.50),
        'q75'             : np.quantile(x, 0.75),
        'q95'             : np.quantile(x, 0.95),
        'q99'             : np.quantile(x, 0.99),
        'q999'            : np.quantile(x, 0.999),
        'max'             : np.max(x),
    }


def to_flat(df, step_cols, max_points=500_000, seed=42):
    """Flatten step_* columns and optionally subsample."""
    vals = df[step_cols].values.ravel().astype(float)
    vals = vals[np.isfinite(vals)]
    if len(vals) > max_points:
        vals = np.random.default_rng(seed).choice(vals, size=max_points, replace=False)
    return vals


def to_increments(df, step_cols, max_points=500_000, seed=42):
    """First differences of each path, flattened and optionally subsampled."""
    arr  = df[step_cols].values.astype(float)
    incs = np.diff(arr, axis=1).ravel()
    incs = incs[np.isfinite(incs)]
    if len(incs) > max_points:
        incs = np.random.default_rng(seed).choice(incs, size=max_points, replace=False)
    return incs


def to_paths_obj(df, step_cols, max_paths=500, seed=42):
    """Object array of 1-D paths for sf.acf / sf.leverage_effect."""
    arr = df[step_cols].values.astype(float)
    n   = len(arr)
    if n > max_paths:
        idx = np.random.default_rng(seed).choice(n, size=max_paths, replace=False)
        arr = arr[idx]
    paths = np.empty(len(arr), dtype=object)
    for i, row in enumerate(arr):
        paths[i] = row[np.isfinite(row)]
    return paths


def fit_powerlaw_fast(x_input, max_sample=100_000, seed=42):
    """Vectorised Hill estimator: minimises KS over candidate x_min values."""
    x = np.abs(np.asarray(x_input, float))
    x = x[np.isfinite(x) & (x > 0)]
    n_total = len(x)
    if n_total < 20:
        return {'alpha': np.nan, 'xmin': np.nan, 'ks': np.nan,
                'n_total': n_total, 'n_tail': 0}
    if n_total > max_sample:
        x = np.random.default_rng(seed).choice(x, size=max_sample, replace=False)
    x       = np.sort(x)
    n       = len(x)
    log_x   = np.log(x)
    rev_cum = np.cumsum(log_x[::-1])[::-1]
    n_tail  = np.arange(n, 0, -1, dtype=np.float64)
    sum_log = rev_cum - n_tail * log_x
    valid   = sum_log > 1e-12
    denom   = np.where(valid, sum_log, 1.0)
    alpha_v = np.where(valid, 1.0 + n_tail / denom, np.inf)
    ks_v    = np.full(n, np.inf)
    for i in range(n - 1):
        if not valid[i]:
            continue
        m      = n - i
        a      = alpha_v[i]
        cdf_th = 1.0 - (x[i] / x[i:]) ** (a - 1.0)
        cdf_em = np.arange(1, m + 1, dtype=np.float64) / m
        ks_v[i] = np.max(np.abs(cdf_em - cdf_th))
    best = int(np.argmin(ks_v))
    return {
        'alpha'  : float(alpha_v[best]),
        'xmin'   : float(x[best]),
        'ks'     : float(ks_v[best]),
        'n_total': n_total,
        'n_tail' : n - best,
    }


def save_fig(fig, name):
    path = Path(OUTPUT_IMAGE_DIR) / f'{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')

## 1. Load Data

EDM values are stored as z-scores and must be unnormalized before comparison with
GARCH and Merton (which are already in log-return units).

Unnormalization: `x = x_norm * std_close + mean_close`

In [4]:
# ── Normalization statistics ───────────────────────────────────────────────────
stats_df = pd.read_csv(UNNORMALIZATION_STATS_PATH, index_col=0)
stats_df.columns = ['mean', 'std']
stats_df.index   = stats_df.index.str.strip().str.lower()
NORM_STATS = stats_df.to_dict('index')
mu_close   = NORM_STATS['close']['mean']
sig_close  = NORM_STATS['close']['std']
print(f'Close unnorm: x = x_norm * {sig_close:.6f} + {mu_close:.6f}')
display(stats_df)

# ── EDM: load generated and reference CSVs ────────────────────────────────────
split_key = 'val' if dataset.lower() in ('val', 'validation') else 'train'
ckpt_dir  = Path(GEN_DIRECTORY)

df_edm_raw = pd.read_csv(ckpt_dir / f'{split_key}_generated_close.csv')
df_ohlc    = pd.read_csv(ckpt_dir / f'{split_key}_gt_ohlc.csv')

EDM_STEP_COLS = [c for c in df_edm_raw.columns if c.startswith('step_')]
SEQ_LEN       = len(EDM_STEP_COLS)

df_edm_raw['ticker'] = df_edm_raw['file'].str.split('_').str[0]
df_ohlc   ['ticker'] = df_ohlc   ['file'].str.split('_').str[0]

# Ground-truth close (normalized; unnormalized below)
df_ref_raw = (
    df_ohlc[df_ohlc['feature'] == 'close']
    .copy().reset_index(drop=True)
)

print(f'\n[EDM {split_key}] generated: {len(df_edm_raw):,} rows | '
      f'windows={df_edm_raw["window_idx"].nunique()} | '
      f'samples/window={df_edm_raw["sample_idx"].nunique()} | '
      f'seq_len={SEQ_LEN}')
print(f'[EDM {split_key}] reference : {len(df_ref_raw):,} windows | '
      f'tickers={df_ref_raw["ticker"].nunique()}')

Close unnorm: x = x_norm * 0.023162 + 0.000482


,mean,std
close,0.000482,0.023162
open,0.000303,0.012624
high,0.013087,0.020523
low,-0.012493,0.020123



[EDM train] generated: 25,000 rows | windows=500 | samples/window=50 | seq_len=512
[EDM train] reference : 500 windows | tickers=262


In [5]:
# Unnormalize EDM generated and reference values
# Both CSVs are in normalized (z-score) units.
df_edm = df_edm_raw.copy()
df_edm[EDM_STEP_COLS] = df_edm_raw[EDM_STEP_COLS].values * sig_close + mu_close

df_ref = df_ref_raw.copy()
df_ref[EDM_STEP_COLS] = df_ref_raw[EDM_STEP_COLS].values * sig_close + mu_close

# Verify scale
for label, df in [('REF (unnorm)', df_ref), ('EDM (unnorm)', df_edm)]:
    v = df[EDM_STEP_COLS].values.ravel()
    v = v[np.isfinite(v)]
    print(f'{label}: mean={v.mean():.6f}  std={v.std():.6f}  '
          f'range=[{v.min():.4f}, {v.max():.4f}]')

REF (unnorm): mean=0.000418  std=0.021991  range=[-0.7580, 0.6258]
EDM (unnorm): mean=0.000382  std=0.019829  range=[-0.5423, 0.3874]


In [6]:
def _load_csv_and_save(csv_path: str, parquet_path: str) -> pd.DataFrame:
    print(f'[csv] Reading {Path(csv_path).name} (slow) …')
    df = pd.read_csv(csv_path, low_memory=False)
    step_cols = [c for c in df.columns if c.startswith('step_')]
    df = df[df['step_000'] != 'step_000'].reset_index(drop=True)
    df[step_cols] = df[step_cols].astype('float32')
    df['ticker'] = df['file'].str.split('_').str[0]
    df.to_parquet(parquet_path, index=False)
    print(f'[csv] Saved cache → {parquet_path}')
    return df


# ── GARCH ─────────────────────────────────────────────────────────────────────
if extract_garch_merton_df:
    df_garch = _load_csv_and_save(GARCH_DIRECTORY, GARCH_PARQUET)
else:
    print(f'[parquet] Loading {GARCH_PARQUET}')
    df_garch = pd.read_parquet(GARCH_PARQUET)

GARCH_STEP_COLS = [c for c in df_garch.columns if c.startswith('step_')]

print(f'[GARCH]  rows={len(df_garch):,}  '
      f'windows={df_garch["window_idx"].nunique()}  '
      f'files={df_garch["file"].nunique()}  '
      f'seq_len={len(GARCH_STEP_COLS)}')

# ── Merton ────────────────────────────────────────────────────────────────────
if extract_garch_merton_df:
    df_merton = _load_csv_and_save(MERTON_DIRECTORY, MERTON_PARQUET)
else:
    print(f'[parquet] Loading {MERTON_PARQUET}')
    df_merton = pd.read_parquet(MERTON_PARQUET)

MERTON_STEP_COLS = [c for c in df_merton.columns if c.startswith('step_')]

print(f'[Merton] rows={len(df_merton):,}  '
      f'windows={df_merton["window_idx"].nunique()}  '
      f'files={df_merton["file"].nunique()}  '
      f'seq_len={len(MERTON_STEP_COLS)}')

[parquet] Loading ./data/generated/GARCH/generated_close.parquet
[GARCH]  rows=295,090  windows=157  files=354  seq_len=512
[parquet] Loading ./data/generated/merton/generated_close.parquet
[Merton] rows=295,090  windows=157  files=354  seq_len=512


In [7]:
# ── Scale diagnostics ─────────────────────────────────────────────────────────
# Expected log-return scale: mean ~ 0.0005, std ~ 0.023.
# Values with std >> 0.5 or pct |x|>1 >> 5 % indicate calibration issues.

model_registry = {
    'Reference': (df_ref,    EDM_STEP_COLS),
    'EDM'      : (df_edm,    EDM_STEP_COLS),
    'GARCH'    : (df_garch,  GARCH_STEP_COLS),
    'Merton'   : (df_merton, MERTON_STEP_COLS),
}

print('=' * 72)
print('  SCALE DIAGNOSTICS  (expected std ~ 0.023 for log-returns)')
print('=' * 72)
for label, (df, cols) in model_registry.items():
    flat = df[cols].values.ravel().astype(float)
    flat = flat[np.isfinite(flat)]
    pct_extreme = (np.abs(flat) > 1.0).mean() * 100
    flag = '  *** SCALE WARNING ***' if np.std(flat) > 0.5 else ''
    print(f'  {label:10s}: n={len(flat):>10,}  '
          f'mean={np.mean(flat):>9.4f}  std={np.std(flat):>8.4f}  '
          f'range=[{np.min(flat):>8.4f}, {np.max(flat):>8.4f}]  '
          f'|x|>1: {pct_extreme:.2f}%{flag}')
print()

  SCALE DIAGNOSTICS  (expected std ~ 0.023 for log-returns)
  Reference : n=   256,000  mean=   0.0004  std=  0.0220  range=[ -0.7580,   0.6258]  |x|>1: 0.00%
  EDM       : n=12,800,000  mean=   0.0004  std=  0.0198  range=[ -0.5423,   0.3874]  |x|>1: 0.00%
  GARCH     : n=151,086,080  mean=   0.0010  std=  0.0250  range=[-21.6370,  28.4118]  |x|>1: 0.00%
  Merton    : n=151,086,080  mean= -35.9092  std=274052.3387  range=[-771831232.0000, 676379840.0000]  |x|>1: 0.12%  *** SCALE WARNING ***



---
## Part 1 — Aggregate and Distributional Comparison

All four datasets are compared on:
- aggregate statistics of generated values
- aggregate statistics of increments (`x_t - x_{t-1}`)
- marginal distribution (KS, Wasserstein-1, histogram, QQ, ECDF)
- stylized facts (fat tails, volatility clustering, leverage effect)
- power-law tail estimates

### 2. Aggregate statistics

In [8]:
flat_data = {
    m: to_flat(df, cols)
    for m, (df, cols) in model_registry.items()
}

df_agg = pd.DataFrame(
    [compute_moments(flat_data[m], m) for m in MODEL_ORDER]
).set_index('model')

print('Aggregate statistics on generated values:')
display(df_agg.T.round(6))

agg_path = Path(OUTPUT_TABLE_DIR) / 'aggregate_statistics.csv'
df_agg.to_csv(agg_path)
print(f'\nSaved: {agg_path}')

Aggregate statistics on generated values:


model,Reference,EDM,GARCH,Merton
n,256000.000000,500000.000000,500000.000000,5.000000e+05
mean,0.000418,0.000362,0.000982,-1.553299e+01
std,0.021991,0.019727,0.029297,2.883625e+04
variance,0.000484,0.000389,0.000858,8.315293e+08
skewness,-0.608519,-0.483665,95.409496,4.240402e+01
excess_kurtosis,39.727241,20.147515,42108.420238,2.888468e+04
IQR,0.017937,0.016893,0.019704,1.973100e-02
min,-0.758010,-0.499762,-5.308519,-5.257112e+06
q001,-0.142796,-0.122878,-0.130828,-3.881810e-01
q01,-0.060820,-0.055770,-0.061778,-6.316700e-02



Saved: tables\generated\classical_comparison\aggregate_statistics.csv


### 3. Aggregate statistics on increments

Increments are defined as `x_t - x_{t-1}` (first difference along the time axis of each path).

In [9]:
inc_data = {
    m: to_increments(df, cols)
    for m, (df, cols) in model_registry.items()
}

df_inc = pd.DataFrame(
    [compute_moments(inc_data[m], m) for m in MODEL_ORDER]
).set_index('model')

print('Aggregate statistics on increments (x_t - x_{t-1}):')
display(df_inc.T.round(6))

inc_path = Path(OUTPUT_TABLE_DIR) / 'increment_statistics.csv'
df_inc.to_csv(inc_path)
print(f'\nSaved: {inc_path}')

Aggregate statistics on increments (x_t - x_{t-1}):


model,Reference,EDM,GARCH,Merton
n,255500.000000,500000.000000,500000.000000,5.000000e+05
mean,0.000006,-0.000035,-0.000008,-2.520959e+01
std,0.031495,0.028070,0.037828,3.931442e+04
variance,0.000992,0.000788,0.001431,1.545624e+09
skewness,0.436824,0.254695,-47.432753,-2.843793e+01
excess_kurtosis,29.386050,15.411283,15187.054435,2.308033e+04
IQR,0.026596,0.024804,0.029695,3.046700e-02
min,-0.732789,-0.482244,-11.007481,-8.325952e+06
q001,-0.195568,-0.165242,-0.172851,-1.042562e+00
q01,-0.085575,-0.077705,-0.089298,-8.772400e-02



Saved: tables\generated\classical_comparison\increment_statistics.csv


### 4. Marginal distribution comparison

- **KS statistic**: maximum absolute difference between empirical CDFs (smaller = more similar).
- **Wasserstein-1**: expected absolute cost to transport one distribution to the other.

In [10]:
ref_flat = flat_data['Reference']

ks_rows = []
for model in ['EDM', 'GARCH', 'Merton']:
    ks_stat, ks_p = scipy_stats.ks_2samp(flat_data[model], ref_flat)
    w1 = scipy_stats.wasserstein_distance(flat_data[model], ref_flat)
    ks_rows.append({'model': model,
                    'KS_stat': round(ks_stat, 4),
                    'KS_pvalue': f'{ks_p:.2e}',
                    'Wasserstein_1': round(w1, 6)})

df_ks = pd.DataFrame(ks_rows).set_index('model')
print('Distributional distances vs Reference:')
display(df_ks)

df_ks.to_csv(Path(OUTPUT_TABLE_DIR) / 'distributional_distances.csv')

Distributional distances vs Reference:


,KS_stat,KS_pvalue,Wasserstein_1
model,,,
EDM,0.0147,2.81e-32,0.000988
GARCH,0.0255,2.25e-96,0.001263
Merton,0.0222,4.74e-73,237.884619


In [11]:
# Histogram + QQ + ECDF
all_vals = np.concatenate([v for v in flat_data.values() if len(v)])
lo, hi   = np.quantile(all_vals, [0.005, 0.995])
bins     = np.linspace(lo, hi, 80)
probs    = np.linspace(0.01, 0.99, 2_000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for model in MODEL_ORDER:
    axes[0].hist(np.clip(flat_data[model], lo, hi), bins=bins,
                 density=True, alpha=0.45, color=MODEL_COLORS[model], label=model)
axes[0].set_xlim(lo, hi)
axes[0].set_title('Marginal distribution of values')
axes[0].set_xlabel('Value (log-return)'); axes[0].set_ylabel('Density')
axes[0].legend()

q_ref = np.quantile(ref_flat, probs)
for model in ['EDM', 'GARCH', 'Merton']:
    q_mod = np.quantile(flat_data[model], probs)
    axes[1].scatter(q_ref, q_mod, s=3, alpha=0.5,
                    label=model, color=MODEL_COLORS[model])
axes[1].plot([lo, hi], [lo, hi], 'k--', lw=1.0, label='y = x')
axes[1].set_xlim(lo, hi); axes[1].set_ylim(lo, hi)
axes[1].set_title('QQ plot vs Reference')
axes[1].set_xlabel('Reference quantiles'); axes[1].set_ylabel('Model quantiles')
axes[1].legend(markerscale=3)

for model in MODEL_ORDER:
    s = np.sort(np.clip(flat_data[model], lo, hi))
    axes[2].plot(s, np.linspace(0, 1, len(s)),
                 color=MODEL_COLORS[model], lw=1.2, label=model)
axes[2].set_xlim(lo, hi)
axes[2].set_title('ECDF comparison')
axes[2].set_xlabel('Value (log-return)'); axes[2].set_ylabel('CDF')
axes[2].legend()

plt.suptitle('Marginal distribution of generated values', y=1.02)
plt.tight_layout()
save_fig(fig, 'marginal_distribution')
display(fig); plt.close(fig)

Saved: images\generated\classical_comparison\marginal_distribution.png


<Figure size 1920x480 with 3 Axes>

### 5. Marginal distribution of increments

In [12]:
ref_inc = inc_data['Reference']

ks_inc_rows = []
for model in ['EDM', 'GARCH', 'Merton']:
    ks_stat, ks_p = scipy_stats.ks_2samp(inc_data[model], ref_inc)
    w1 = scipy_stats.wasserstein_distance(inc_data[model], ref_inc)
    ks_inc_rows.append({'model': model,
                        'KS_stat': round(ks_stat, 4),
                        'KS_pvalue': f'{ks_p:.2e}',
                        'Wasserstein_1': round(w1, 6)})

df_ks_inc = pd.DataFrame(ks_inc_rows).set_index('model')
print('Distributional distances on increments vs Reference:')
display(df_ks_inc)
df_ks_inc.to_csv(Path(OUTPUT_TABLE_DIR) / 'increment_distributional_distances.csv')

all_incs   = np.concatenate([v for v in inc_data.values() if len(v)])
lo_i, hi_i = np.quantile(all_incs, [0.005, 0.995])
bins_i     = np.linspace(lo_i, hi_i, 80)
probs_i    = np.linspace(0.01, 0.99, 2_000)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for model in MODEL_ORDER:
    axes[0].hist(np.clip(inc_data[model], lo_i, hi_i), bins=bins_i,
                 density=True, alpha=0.45, color=MODEL_COLORS[model], label=model)
axes[0].set_xlim(lo_i, hi_i)
axes[0].set_title('Marginal distribution of increments')
axes[0].set_xlabel('x_t - x_{t-1}'); axes[0].set_ylabel('Density')
axes[0].legend()

q_ref_i = np.quantile(ref_inc, probs_i)
for model in ['EDM', 'GARCH', 'Merton']:
    q_mod_i = np.quantile(inc_data[model], probs_i)
    axes[1].scatter(q_ref_i, q_mod_i, s=3, alpha=0.5,
                    label=model, color=MODEL_COLORS[model])
axes[1].plot([lo_i, hi_i], [lo_i, hi_i], 'k--', lw=1.0, label='y = x')
axes[1].set_xlim(lo_i, hi_i); axes[1].set_ylim(lo_i, hi_i)
axes[1].set_title('QQ plot — Increments vs Reference')
axes[1].set_xlabel('Reference quantiles'); axes[1].set_ylabel('Model quantiles')
axes[1].legend(markerscale=3)

for model in MODEL_ORDER:
    s = np.sort(np.clip(inc_data[model], lo_i, hi_i))
    axes[2].plot(s, np.linspace(0, 1, len(s)),
                 color=MODEL_COLORS[model], lw=1.2, label=model)
axes[2].set_xlim(lo_i, hi_i)
axes[2].set_title('ECDF — Increments')
axes[2].set_xlabel('x_t - x_{t-1}'); axes[2].set_ylabel('CDF')
axes[2].legend()

plt.suptitle('Marginal distribution of increments', y=1.02)
plt.tight_layout()
save_fig(fig, 'increment_distribution')
display(fig); plt.close(fig)

Distributional distances on increments vs Reference:


,KS_stat,KS_pvalue,Wasserstein_1
model,,,
EDM,0.0149,3.89e-33,0.001590
GARCH,0.0240,8.22e-85,0.002182
Merton,0.0275,1.95e-111,367.345705


Saved: images\generated\classical_comparison\increment_distribution.png


<Figure size 1920x480 with 3 Axes>

### 6. Stylized facts

Computed via `replication.stylized_facts` — the same interface used throughout the project.

- **`sf.distribution`**: normalized (z-scored) PDF of return values — reveals fat tails.
- **`sf.acf`**: mean ACF of `|r_t|` across paths — reveals volatility clustering.
- **`sf.leverage_effect`**: `L(t) = Corr(r_s, |r_{s+t}|^2)` — negative for real equities.

Each model is plotted individually (saved to `OUTPUT_IMAGE_DIR`) and then overlaid for comparison.

> **Note:** `sf.acf` computes autocorrelation lag-by-lag via `pandas.autocorr`; it is slow for large
> path counts. `MAX_SF_PATHS` and `MAX_LAG_ACF` cap the computation.

In [13]:
MAX_SF_PATHS = 500
MAX_LAG_ACF  = min(100, SEQ_LEN // 2)
MAX_LAG_LEV  = min(30, SEQ_LEN // 2)

sf_flat  = {m: to_flat(df, cols, max_points=500_000)
            for m, (df, cols) in model_registry.items()}
sf_paths = {m: to_paths_obj(df, cols, max_paths=MAX_SF_PATHS)
            for m, (df, cols) in model_registry.items()}

print('Path counts for stylized facts:')
for m, arr in sf_paths.items():
    lens = [len(p) for p in arr]
    print(f'  {m}: {len(arr)} paths, len [{min(lens)}, {max(lens)}]')

Path counts for stylized facts:
  Reference: 500 paths, len [512, 512]
  EDM: 500 paths, len [512, 512]
  GARCH: 500 paths, len [512, 512]
  Merton: 500 paths, len [512, 512]


In [14]:
# sf.distribution — fat-tail PDF (normalizes internally)
dist_results = {}
for model in MODEL_ORDER:
    out = str(Path(OUTPUT_IMAGE_DIR) / f'sf_distribution_{model}')
    dx, dy = sf.distribution(
        sf_flat[model], file_name=out,
        scale='log', multiple=False, normalize=True, granuality=100,
    )
    dist_results[model] = (dx, dy)
    print(f'  {model}: sf.distribution saved')

  Reference: sf.distribution saved
  EDM: sf.distribution saved
  GARCH: sf.distribution saved
  Merton: sf.distribution saved


In [15]:
# sf.acf — volatility clustering (ACF of |r_t|)
acf_results = {}
for model in MODEL_ORDER:
    out = str(Path(OUTPUT_IMAGE_DIR) / f'sf_acf_{model}')
    res = sf.acf(
        sf_paths[model], file_name=out,
        for_abs=True, multiple=True, fit=False,
        scale='log', max_lag=MAX_LAG_ACF,
    )
    acf_results[model] = res
    print(f'  {model}: sf.acf saved  (shape={res.shape})')

  Reference: sf.acf saved  (shape=(100,))
  EDM: sf.acf saved  (shape=(100,))
  GARCH: sf.acf saved  (shape=(100,))
  Merton: sf.acf saved  (shape=(100,))


In [16]:
# sf.leverage_effect — L(t) should be negative for equities
lev_results = {}
for model in MODEL_ORDER:
    out = str(Path(OUTPUT_IMAGE_DIR) / f'sf_leverage_{model}')
    res = sf.leverage_effect(
        sf_paths[model], file_name=out,
        multiple=True, min_lag=1, max_lag=MAX_LAG_LEV,
    )
    lev_results[model] = res
    print(f'  {model}: sf.leverage_effect saved  (shape={res.shape})')

  Reference: sf.leverage_effect saved  (shape=(29,))
  EDM: sf.leverage_effect saved  (shape=(29,))
  GARCH: sf.leverage_effect saved  (shape=(29,))
  Merton: sf.leverage_effect saved  (shape=(29,))


In [17]:
# Overlap comparison — all models on the same axes
lag_axis = np.arange(1, MAX_LAG_ACF + 1)
lev_lags = np.arange(1, MAX_LAG_LEV)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Fat-tail distribution (positive side, log-log) ---
ax = axes[0]
for model in MODEL_ORDER:
    dx, dy = dist_results[model]
    mask = dx > 0
    ax.plot(dx[mask], dy[mask], '.', ms=3,
            color=MODEL_COLORS[model], alpha=0.8, label=model)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Normalised scale (sigma)')
ax.set_ylabel('PDF P(r)')
ax.set_title('Fat-tail distribution (log-log, positive side)')
ax.legend()

# --- ACF of |r_t| ---
ax = axes[1]
for model in MODEL_ORDER:
    acf_vals = np.abs(acf_results[model])  # ACF may be slightly negative; abs for log scale
    ax.plot(lag_axis, acf_vals, '.', ms=3,
            color=MODEL_COLORS[model], alpha=0.8, label=model)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Lag k')
ax.set_ylabel('ACF of |r_t|')
ax.set_title('Volatility clustering — ACF overlap')
ax.legend()

# --- Leverage effect ---
ax = axes[2]
for model in MODEL_ORDER:
    ax.plot(lev_lags, lev_results[model],
            color=MODEL_COLORS[model], lw=1.4, label=model)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Lag t')
ax.set_ylabel('L(t)')
ax.set_title('Leverage effect — all models')
ax.legend()

plt.suptitle('Stylized facts comparison — all models', y=1.02)
plt.tight_layout()
save_fig(fig, 'stylized_facts_overlap')
display(fig); plt.close(fig)

Saved: images\generated\classical_comparison\stylized_facts_overlap.png


<Figure size 2160x600 with 3 Axes>

### 7. Power-law tail estimates

We apply the fast **Hill estimator** (minimises KS distance over candidate `x_min` values):

    alpha_hat = 1 + n_tail / sum_{i} log(x_i / x_min)

Smaller `alpha` = heavier tail. Real equity return tails typically give `alpha ~ 3–5`.

In [18]:
pl_rows = []
for model in MODEL_ORDER:
    res = fit_powerlaw_fast(flat_data[model])
    res['model'] = model
    pl_rows.append(res)
    print(f'{model:10s}: alpha={res["alpha"]:.4f}  '
          f'xmin={res["xmin"]:.6f}  '
          f'KS={res["ks"]:.4f}  '
          f'n_tail={res["n_tail"]:,}')

df_pl = pd.DataFrame(pl_rows).set_index('model')
display(df_pl.round(4))
df_pl.to_csv(Path(OUTPUT_TABLE_DIR) / 'powerlaw_estimates.csv')

Reference : alpha=3.4849  xmin=0.044633  KS=0.0200  n_tail=4,173
EDM       : alpha=3.7876  xmin=0.048246  KS=0.0229  n_tail=2,853
GARCH     : alpha=4.1689  xmin=0.071445  KS=0.0135  n_tail=1,445
Merton    : alpha=2.7666  xmin=0.018784  KS=0.0344  n_tail=24,193


,alpha,xmin,ks,n_total,n_tail
model,,,,,
Reference,3.4849,0.0446,0.0200,256000,4173
EDM,3.7876,0.0482,0.0229,500000,2853
GARCH,4.1689,0.0714,0.0135,499949,1445
Merton,2.7666,0.0188,0.0344,500000,24193


---
## Part 2 — Matched-window conditional comparison

Windows are matched on `(ticker, window_start)`, where `ticker` is extracted from the
filename and `window_start` is the row offset in the source CSV.

**Caveat:** this alignment is valid only if all models were generated from the **same
source directory** (same date range). If source folders differ (e.g., post-2002 vs.
full history), `window_start` row indices refer to different calendar dates and the
matching will yield zero common windows — reported clearly below.

In [19]:
def add_key(df):
    df = df.copy()
    df['_key'] = list(zip(df['ticker'], df['window_start']))
    return df

df_ref_k    = add_key(df_ref)
df_edm_k    = add_key(df_edm)
df_garch_k  = add_key(df_garch)
df_merton_k = add_key(df_merton)

keys_ref    = set(df_ref_k   ['_key'])
keys_edm    = set(df_edm_k   ['_key'])
keys_garch  = set(df_garch_k ['_key'])
keys_merton = set(df_merton_k['_key'])

common_keys = keys_ref & keys_edm & keys_garch & keys_merton

print(f'Keys in Reference : {len(keys_ref):,}')
print(f'Keys in EDM       : {len(keys_edm):,}')
print(f'Keys in GARCH     : {len(keys_garch):,}')
print(f'Keys in Merton    : {len(keys_merton):,}')
print(f'Common keys       : {len(common_keys):,}')

if not common_keys:
    print()
    print('=' * 70)
    print('  NO COMMON WINDOWS FOUND')
    print('  Likely cause: models were run on different input folders.')
    print('  Rerun GARCH and Merton on the same source directory as EDM,')
    print('  then re-execute Part 2.')
    print('=' * 70)

Keys in Reference : 500
Keys in EDM       : 500
Keys in GARCH     : 29,509
Keys in Merton    : 29,509
Common keys       : 500


In [20]:
if common_keys:
    common_tickers = {k[0] for k in common_keys}
    print(f'Matched tickers    : {len(common_tickers)}')
    print(f'Sample tickers     : {sorted(common_tickers)[:10]}')
    print(f'window_start range : [{min(k[1] for k in common_keys)}, '
          f'{max(k[1] for k in common_keys)}]')

    for lbl, df_k in [('EDM', df_edm_k), ('GARCH', df_garch_k), ('Merton', df_merton_k)]:
        sub = df_k[df_k['_key'].isin(common_keys)]
        print(f'  {lbl}: {len(sub):,} rows  '
              f'samples/window={sub["sample_idx"].nunique()}')

    print(f'\nDropped windows (not in all sources):')
    print(f'  Reference: {len(keys_ref    - common_keys):,}')
    print(f'  EDM      : {len(keys_edm    - common_keys):,}')
    print(f'  GARCH    : {len(keys_garch  - common_keys):,}')
    print(f'  Merton   : {len(keys_merton - common_keys):,}')
else:
    print('Skipping — no common windows.')

Matched tickers    : 262
Sample tickers     : ['A', 'AAPL', 'ACGL', 'ACN', 'ADI', 'ADM', 'ADP', 'AEE', 'AEP', 'AES']
window_start range : [0, 5600]
  EDM: 25,000 rows  samples/window=50
  GARCH: 5,000 rows  samples/window=10
  Merton: 5,000 rows  samples/window=10

Dropped windows (not in all sources):
  Reference: 0
  EDM      : 0
  GARCH    : 29,009
  Merton   : 29,009


In [21]:
# ── Per-window metrics: MAE, RMSE, bias ──────────────────────────────────────
# Ensemble mean used as the point forecast.

df_metrics    = pd.DataFrame()
df_summary_pw = pd.DataFrame()

if common_keys:
    MIN_SEQ   = min(len(EDM_STEP_COLS), len(GARCH_STEP_COLS), len(MERTON_STEP_COLS))
    CMP_STEPS = [f'step_{t:03d}' for t in range(MIN_SEQ)]

    gen_frames = [
        ('EDM',    df_edm_k,    CMP_STEPS),
        ('GARCH',  df_garch_k,  CMP_STEPS),
        ('Merton', df_merton_k, CMP_STEPS),
    ]

    metric_rows = []
    for key in sorted(common_keys):
        ref_row = df_ref_k[df_ref_k['_key'] == key]
        if len(ref_row) == 0:
            continue
        gt = ref_row[CMP_STEPS].values[0].astype(float)

        for model_lbl, df_mod, steps in gen_frames:
            gen = df_mod[df_mod['_key'] == key][steps].values.astype(float)
            if gen.shape[0] == 0:
                continue
            gm  = np.nanmean(gen, axis=0)
            metric_rows.append({
                'model'       : model_lbl,
                'ticker'      : key[0],
                'window_start': key[1],
                'MAE'         : float(np.nanmean(np.abs(gm - gt))),
                'RMSE'        : float(np.sqrt(np.nanmean((gm - gt) ** 2))),
                'bias'        : float(np.nanmean(gm - gt)),
            })

    df_metrics = pd.DataFrame(metric_rows)
    df_summary_pw = (
        df_metrics.groupby('model')[['MAE', 'RMSE', 'bias']]
        .agg(['mean', 'median', 'std'])
        .round(6)
    )
    print('Per-window metrics (ensemble mean vs GT):')
    display(df_summary_pw)
    df_metrics.to_csv(Path(OUTPUT_TABLE_DIR) / 'matched_window_metrics.csv', index=False)
else:
    print('Skipping per-window metrics — no common windows.')

Per-window metrics (ensemble mean vs GT):


MAE                                RMSE            \
              mean    median          std         mean    median   
model                                                              
EDM       0.004295  0.003806     0.002201     0.006051  0.005278   
GARCH     0.015007  0.013565     0.006303     0.021127  0.019207   
Merton  394.854272  0.013647  4211.451072  7217.525608  0.019285   

                           bias                         
                 std       mean    median          std  
model                                                   
EDM         0.003478  -0.000036 -0.000020     0.000440  
GARCH       0.009599   0.000611  0.000581     0.000978  
Merton  79754.531817 -25.147796  0.000003  3585.159249

### CRPS — Continuous Ranked Probability Score

    CRPS(F, y) = E[|X - y|] - 0.5 * E[|X - X'|]

Lower CRPS is better. For a perfect point forecast CRPS = MAE; the gap
`MAE - CRPS` measures how much the ensemble spread contributes beyond the mean prediction.

In [22]:
def crps_ensemble(samples, obs):
    """CRPS for a single observation given an ensemble of samples."""
    s = np.asarray(samples, float)
    s = s[np.isfinite(s)]
    if len(s) == 0:
        return np.nan
    term1 = np.mean(np.abs(s - obs))
    term2 = np.mean(np.abs(s[:, None] - s[None, :])) * 0.5
    return term1 - term2


df_crps         = pd.DataFrame()
df_crps_summary = pd.DataFrame()

if common_keys:
    crps_rows = []
    for key in sorted(common_keys):
        ref_row = df_ref_k[df_ref_k['_key'] == key]
        if len(ref_row) == 0:
            continue
        gt = ref_row[CMP_STEPS].values[0].astype(float)

        for model_lbl, df_mod, steps in gen_frames:
            gen = df_mod[df_mod['_key'] == key][steps].values.astype(float)
            if gen.shape[0] == 0:
                continue
            crps_t = np.array([
                crps_ensemble(gen[:, t], gt[t]) for t in range(MIN_SEQ)
            ])
            crps_rows.append({
                'model'       : model_lbl,
                'ticker'      : key[0],
                'window_start': key[1],
                'mean_CRPS'   : float(np.nanmean(crps_t)),
            })

    if crps_rows:
        df_crps = pd.DataFrame(crps_rows)
        df_crps_summary = (
            df_crps.groupby('model')['mean_CRPS']
            .agg(['mean', 'median', 'std'])
            .round(6)
        )
        print('CRPS summary (lower is better):')
        display(df_crps_summary)
        df_crps.to_csv(Path(OUTPUT_TABLE_DIR) / 'crps_by_model.csv', index=False)
    else:
        print('No CRPS results.')
else:
    print('Skipping CRPS — no common windows.')

CRPS summary (lower is better):


,mean,median,std
model,,,
EDM,0.002891,0.002589,0.001470
GARCH,0.011524,0.010426,0.005096
Merton,149.228696,0.010440,1124.217996


### Interval coverage — calibration check

For each nominal level `1-alpha`, the empirical interval `[q_{alpha/2}, q_{1-alpha/2}]` of
the ensemble is computed per window, and we check whether the reference falls inside.

- `gap > 0` → underconfident (intervals too wide)
- `gap < 0` → overconfident (intervals too narrow)

In [23]:
COVERAGE_LEVELS = [0.50, 0.80, 0.90, 0.95, 0.99]

df_cov         = pd.DataFrame()
df_cov_summary = pd.DataFrame()

if common_keys:
    cov_rows = []
    for key in sorted(common_keys):
        ref_row = df_ref_k[df_ref_k['_key'] == key]
        if len(ref_row) == 0:
            continue
        gt = ref_row[CMP_STEPS].values[0].astype(float)

        for model_lbl, df_mod, steps in gen_frames:
            gen = df_mod[df_mod['_key'] == key][steps].values.astype(float)
            if gen.shape[0] == 0:
                continue
            for level in COVERAGE_LEVELS:
                a2 = (1 - level) / 2
                lo_q = np.quantile(gen, a2,       axis=0)
                hi_q = np.quantile(gen, 1 - a2,   axis=0)
                emp  = float(np.mean((gt >= lo_q) & (gt <= hi_q)))
                wid  = float(np.mean(hi_q - lo_q))
                cov_rows.append({
                    'model'       : model_lbl,
                    'ticker'      : key[0],
                    'window_start': key[1],
                    'nominal'     : level,
                    'empirical'   : emp,
                    'mean_width'  : wid,
                })

    if cov_rows:
        df_cov = pd.DataFrame(cov_rows)
        df_cov_summary = (
            df_cov.groupby(['model', 'nominal'])[['empirical', 'mean_width']]
            .mean()
            .round(4)
        )
        df_cov_summary['gap'] = (
            df_cov_summary['empirical']
            - df_cov_summary.index.get_level_values('nominal')
        ).round(4)
        print('Interval coverage:')
        display(df_cov_summary)
        df_cov_summary.to_csv(Path(OUTPUT_TABLE_DIR) / 'interval_coverage.csv')

        # Calibration curve
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot([0, 1], [0, 1], 'k--', lw=1.0, label='Perfect calibration')
        for model in ['EDM', 'GARCH', 'Merton']:
            try:
                sub = df_cov_summary.xs(model, level='model')
                ax.plot(sub.index, sub['empirical'], 'o-', lw=1.5,
                        color=MODEL_COLORS[model], label=model)
            except KeyError:
                pass
        ax.set_xlabel('Nominal coverage'); ax.set_ylabel('Empirical coverage')
        ax.set_title('Coverage calibration curve')
        ax.legend(); ax.grid(True, lw=0.3)
        plt.tight_layout()
        save_fig(fig, 'coverage_calibration')
        display(fig); plt.close(fig)
    else:
        print('No coverage results.')
else:
    print('Skipping coverage — no common windows.')

Interval coverage:


empirical  mean_width     gap
model  nominal                               
EDM    0.50        0.4330      0.0066 -0.0670
       0.80        0.7350      0.0123 -0.0650
       0.90        0.8512      0.0153 -0.0488
       0.95        0.9105      0.0174 -0.0395
       0.99        0.9572      0.0202 -0.0328
GARCH  0.50        0.4449      0.0192 -0.0551
       0.80        0.6977      0.0389 -0.1023
       0.90        0.7865      0.0528 -0.1135
       0.95        0.8119      0.0598 -0.1381
       0.99        0.8280      0.0654 -0.1620
Merton 0.50        0.4482    475.5308 -0.0518
       0.80        0.6976   1134.9767 -0.1024
       0.90        0.7869   2634.0913 -0.1131
       0.95        0.8137   3383.6486 -0.1363
       0.99        0.8303   3983.2945 -0.1597

Saved: images\generated\classical_comparison\coverage_calibration.png


<Figure size 720x600 with 1 Axes>

---
## 8. Summary tables

All tables have been saved as CSV to `OUTPUT_TABLE_DIR`. This cell collects the key
results for a final thesis-ready overview.

In [24]:
print('Output tables in:', Path(OUTPUT_TABLE_DIR).resolve())
for f in sorted(Path(OUTPUT_TABLE_DIR).glob('*.csv')):
    print(f'  {f.name}')

print()
print('=' * 70)
print('  AGGREGATE STATISTICS (key columns)')
print('=' * 70)
display(df_agg[['mean', 'std', 'skewness', 'excess_kurtosis', 'q01', 'q99']].round(6))

print()
print('=' * 70)
print('  DISTRIBUTIONAL DISTANCES vs REFERENCE')
print('=' * 70)
display(df_ks)

print()
print('=' * 70)
print('  POWER-LAW TAIL ESTIMATES')
print('=' * 70)
display(df_pl[['alpha', 'xmin', 'ks', 'n_tail']].round(4))

if not df_summary_pw.empty:
    print()
    print('=' * 70)
    print('  MATCHED-WINDOW METRICS (ENSEMBLE MEAN vs GT)')
    print('=' * 70)
    display(df_summary_pw)

if not df_crps_summary.empty:
    print()
    print('=' * 70)
    print('  CRPS SUMMARY (LOWER IS BETTER)')
    print('=' * 70)
    display(df_crps_summary)

if not df_cov_summary.empty:
    print()
    print('=' * 70)
    print('  INTERVAL COVERAGE')
    print('=' * 70)
    display(df_cov_summary)

Output tables in: C:\Users\Lenovo\Documents\SCUOLA\UNI\MASTER\2ANNO\THESIS\Master-Thesis\tables\generated\classical_comparison
  aggregate_statistics.csv
  crps_by_model.csv
  distributional_distances.csv
  increment_distributional_distances.csv
  increment_statistics.csv
  interval_coverage.csv
  matched_window_metrics.csv
  powerlaw_estimates.csv

  AGGREGATE STATISTICS (key columns)


,mean,std,skewness,excess_kurtosis,q01,q99
model,,,,,,
Reference,0.000418,0.021991,-0.608519,39.727241,-0.060820,0.059391
EDM,0.000362,0.019727,-0.483665,20.147515,-0.055770,0.054351
GARCH,0.000982,0.029297,95.409496,42108.420238,-0.061778,0.064232
Merton,-15.532993,28836.249088,42.404021,28884.680369,-0.063167,0.064890



  DISTRIBUTIONAL DISTANCES vs REFERENCE


,KS_stat,KS_pvalue,Wasserstein_1
model,,,
EDM,0.0147,2.81e-32,0.000988
GARCH,0.0255,2.25e-96,0.001263
Merton,0.0222,4.74e-73,237.884619



  POWER-LAW TAIL ESTIMATES


...

MAE                                RMSE            \
              mean    median          std         mean    median   
model                                                              
EDM       0.004295  0.003806     0.002201     0.006051  0.005278   
GARCH     0.015007  0.013565     0.006303     0.021127  0.019207   
Merton  394.854272  0.013647  4211.451072  7217.525608  0.019285   

                           bias                         
                 std       mean    median          std  
model                                                   
EDM         0.003478  -0.000036 -0.000020     0.000440  
GARCH       0.009599   0.000611  0.000581     0.000978  
Merton  79754.531817 -25.147796  0.000003  3585.159249


  CRPS SUMMARY (LOWER IS BETTER)


,mean,median,std
model,,,
EDM,0.002891,0.002589,0.001470
GARCH,0.011524,0.010426,0.005096
Merton,149.228696,0.010440,1124.217996



  INTERVAL COVERAGE


empirical  mean_width     gap
model  nominal                               
EDM    0.50        0.4330      0.0066 -0.0670
       0.80        0.7350      0.0123 -0.0650
       0.90        0.8512      0.0153 -0.0488
       0.95        0.9105      0.0174 -0.0395
       0.99        0.9572      0.0202 -0.0328
GARCH  0.50        0.4449      0.0192 -0.0551
       0.80        0.6977      0.0389 -0.1023
       0.90        0.7865      0.0528 -0.1135
       0.95        0.8119      0.0598 -0.1381
       0.99        0.8280      0.0654 -0.1620
Merton 0.50        0.4482    475.5308 -0.0518
       0.80        0.6976   1134.9767 -0.1024
       0.90        0.7869   2634.0913 -0.1131
       0.95        0.8137   3383.6486 -0.1363
       0.99        0.8303   3983.2945 -0.1597

In [25]:
import pandas as pd
df = pd.read_csv('./checkpoints/merton/manifest.csv')
print(df[['ticker','mu','a0','lambda','sigma_J']].head(20))
print(df['mu'].describe())

                        ticker        mu        a0         lambda  \
0   AAPL_1980-12-12_2026-05-01  0.000683 -8.399012   4.642702e-01   
1   ABBV_2013-01-02_2026-05-01  0.001491 -9.096542   2.475389e-01   
2    ABT_1980-03-17_2026-05-01  0.000573 -9.011904   2.886116e-01   
3   ACGL_1995-09-14_2026-05-01  0.000903 -9.449768   3.824908e-01   
4    ACN_2001-07-19_2026-05-01  0.000865 -9.125951   3.051729e-01   
5   ADBE_1986-08-13_2026-05-01  0.000698 -8.572365   0.000000e+00   
6    ADI_1980-03-17_2026-05-01 -0.000167 -8.450375   4.791876e-01   
7    ADM_1980-03-17_2026-05-01  0.000545 -8.590473   1.819409e-01   
8    ADP_1980-03-17_2026-05-01  0.000525 -9.261207   3.456493e-01   
9   ADSK_1985-06-28_2026-05-01  0.000893 -8.526510   9.451294e-08   
10   AEE_1998-01-02_2026-05-01  0.000779 -9.309320   1.175268e-01   
11   AEP_1962-01-02_2026-05-01  0.000381 -9.410874   1.585267e-01   
12   AES_1991-06-26_2026-05-01  0.000780 -8.529855   0.000000e+00   
13   AFL_1980-03-17_2026-05-01  0.

In [26]:
import pandas as pd, os
f = sorted(f for f in os.listdir('./data/SNP500_individual_processed_copy') if f.endswith('.csv'))[0]
df = pd.read_csv(f'./data/SNP500_individual_processed_copy/{f}')
print(df[['open','high','low','close']].describe())
print(df[['open','high','low','close']].head())


               open          high           low         close
count  1.143700e+04  11437.000000  11437.000000  11437.000000
mean   9.281164e-04      0.015974     -0.014587      0.000696
std    1.671208e-02      0.022556      0.024150      0.027979
min   -6.408034e-01     -0.612386     -0.745917     -0.731248
25%   -3.812492e-03      0.004439     -0.023404     -0.012474
50%    2.916617e-07      0.011941     -0.010561      0.000000
75%    5.871915e-03      0.023931     -0.002328      0.014043
max    2.456724e-01      0.340085      0.235723      0.286892
       open      high       low     close
0 -0.049004 -0.049004 -0.053581 -0.053581
1 -0.071293 -0.071293 -0.076231 -0.076231
2  0.024449  0.029268  0.024449  0.024449
3  0.028580  0.033264  0.028580  0.028580
4  0.059239  0.063654  0.059239  0.059239


In [27]:
import pandas as pd
df = pd.read_csv('./data/generated/merton/generated_close.csv', nrows=5)
print(df[['window_idx','sample_idx','file','window_start'] + [c for c in df.columns if c.startswith('step_')][:10]])


   window_idx  sample_idx                            file  window_start  \
0           0           0  AAPL_1980-12-12_2026-05-01.csv             0   
1           0           1  AAPL_1980-12-12_2026-05-01.csv             0   
2           0           2  AAPL_1980-12-12_2026-05-01.csv             0   
3           0           3  AAPL_1980-12-12_2026-05-01.csv             0   
4           0           4  AAPL_1980-12-12_2026-05-01.csv             0   

   step_000  step_001  step_002  step_003  step_004  step_005  step_006  \
0  0.005255 -0.014921  0.019314  0.014794 -0.028589 -0.045206  0.002601   
1  0.002175 -0.000609  0.012547  0.005854  0.010710 -0.059724  0.024372   
2 -0.026773 -0.019092 -0.043123  0.013077  0.000613 -0.011195  0.018501   
3  0.023868 -0.024634 -0.013274 -0.017297 -0.003231 -0.020411  0.004238   
4  0.002217  0.015466 -0.001704 -0.049025  0.018071  0.122101 -0.023808   

   step_007  step_008  step_009  
0 -0.004062 -0.029877 -0.043846  
1  0.000584 -0.013873 -0.01263

In [28]:
import numpy as np

theta = np.load('./checkpoints/merton/AAPL_1980-12-12_2026-05-01_theta.npy')
print('Raw theta:', theta)
print(f'mu      = {theta[0]:.6f}')
print(f'a0      = {theta[1]:.6f}')
print(f'a[0]    = {theta[2]:.4f}')
print(f'a[1]    = {theta[3]:.4f}')
print(f'lambda  = {np.exp(theta[4]):.6f}')
print(f'mu_J    = {theta[5]:.6f}')
print(f'sigma_J = {np.exp(theta[6]):.6f}')


Raw theta: [ 6.82916536e-04 -8.39901182e+00  1.81345783e-02  6.05112233e-02
 -7.67288478e-01  2.73316404e-05 -3.42629041e+00]
mu      = 0.000683
a0      = -8.399012
a[0]    = 0.0181
a[1]    = 0.0605
lambda  = 0.464270
mu_J    = 0.000027
sigma_J = 0.032507


In [29]:
import numpy as np, pandas as pd

theta = np.load('./checkpoints/merton/AAPL_1980-12-12_2026-05-01_theta.npy')

def unpack_params(theta, k_var):
    idx = 0
    mu = theta[idx]; idx += 1
    a0 = theta[idx]; idx += 1
    a = theta[idx:idx+k_var]; idx += k_var
    lam = np.exp(theta[idx]); idx += 1
    mu_J = theta[idx]; idx += 1
    sigma_J = np.exp(theta[idx]); idx += 1
    return mu, a0, a, lam, mu_J, sigma_J

mu, a0, a, lam, mu_J, sigma_J = unpack_params(theta, k_var=2)

df = pd.read_csv('./data/SNP500_individual_processed_copy/AAPL_1980-12-12_2026-05-01.csv')
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
df = df.sort_values("date").reset_index(drop=True)

Q_w = np.column_stack([df["open"].values[:512] ** 2,
                       (df["high"].values[:512] - df["low"].values[:512]) ** 2])

log_nu2 = np.clip(a0 + Q_w @ a, -30, 30)
nu = np.sqrt(np.exp(log_nu2))
print(f"nu: mean={nu.mean():.5f}  min={nu.min():.5f}  max={nu.max():.5f}")

rng = np.random.default_rng(42)
Z = rng.standard_normal((5, 512))
n_jumps = rng.poisson(lam, size=(5, 512))
has_jump = n_jumps > 0
jump_std = np.where(has_jump, np.sqrt(n_jumps.astype(float)) * sigma_J, 1.0)
jump_sum = np.where(has_jump, rng.normal(n_jumps * mu_J, jump_std), 0.0)
paths = mu + nu[None, :] * Z + jump_sum

print("\nFirst 10 steps, 5 samples:")
print(paths[:, :10].round(6))
print(f"\nPath stats: mean={paths.mean():.5f}  std={paths.std():.5f}")


nu: mean=0.01500  min=0.01500  max=0.01501

First 10 steps, 5 samples:
[[ 0.004183 -0.049307  0.011942  0.014794 -0.028589 -0.018854  0.002601
  -0.004062  0.000431  0.018901]
 [ 0.002175 -0.000609  0.012547  0.005854  0.01071  -0.009645  0.014153
   0.038902 -0.013873 -0.012635]
 [-0.030943 -0.004785 -0.032027  0.001224  0.011618  0.016369  0.043679
   0.003725 -0.001363 -0.036455]
 [ 0.011924  0.023052 -0.013274  0.024326 -0.003231  0.00665   0.004238
  -0.000892  0.021146 -0.012173]
 [ 0.002217 -0.020206  0.012188 -0.017691 -0.016681  0.065061  0.006181
   0.009431  0.048801 -0.008191]]

Path stats: mean=-0.00012  std=0.02509
